In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim import Adam
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# transform to tensors
X_train = torch.tensor(X_train.values, dtype=torch.float32)
X_test  = torch.tensor(X_test.values, dtype=torch.float32)
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test  = torch.tensor(y_test.values, dtype=torch.long)


# Prepare the dataset
X_train = labels_df['X_train'].reshape(labels_df['X_train'].shape[0], -1)  # Flatten for fully connected layers
y_train = labels_df['y_train'].long()
X_test = labels_df['X_test'].reshape(labels_df['X_test'].shape[0], -1)
y_test = labels_df['y_test'].long()




In [ ]:

# Create TensorDatasets for training and testing
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

# Create Dataloaders to train and test data in batches
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Print dataset sizes
print("Train dataset:", len(train_dataset))
print("Test dataset:", len(test_dataset))

# Get the first batch from the training DataLoader
X_batch, y_batch = next(iter(train_loader))
print(f"Training batch input shape: {X_batch.shape}")
print(f"Training batch labels shape: {y_batch.shape}")

In [ ]:
# 5. Display sample images



In [ ]:
# Task 1: Write your model class here:

class NN4Layer(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim):
        super(NN4Layer, self).__init__()

        self.layer1 = nn.Linear(input_dim, hidden_dim)

        self.layer2 = nn.Linear(hidden_dim, hidden_dim)

        self.layer3 = nn.Linear(hidden_dim, hidden_dim)

        self.layer4 = nn.Linear(hidden_dim, output_dim)

        # ReLU activation for non-linearity
        self.relu = nn.ReLU()

    def forward(self, x):
        z1 = self.layer1(x)
        a1 = self.relu(z1)

        z2 = self.layer2(a1)
        a2 = self.relu(z2)

        z3 = self.layer3(a2)
        a3 = self.relu(z3)

        output = self.layer4(a3)

        return output


In [ ]:
# Task 2: Write your training loop here:

def train_one_epoch(model, optimizer, criterion, train_loader, device):
    model.train()

    running_loss = 0.0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.view(X_batch.size(0), -1).to(device)
        y_batch = y_batch.to(device)

        #Forward pass
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        # Backward pass & optimization
        optimizer.zero_grad()   # to clear previous gradients
        loss.backward()         # to compute gradients
        optimizer.step()        # to update model parameters

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    return avg_loss

In [ ]:
# Task 3: Write your validation loop here:

def validate(model, criterion, test_loader, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:

            # to move data to device
            X_batch = X_batch.view(X_batch.size(0), -1).to(device)
            y_batch = y_batch.to(device)

            # Forward pass
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            running_loss += loss.item()

            probabilities = F.softmax(outputs, dim=1)

            predicted = torch.argmax(probabilities, dim=1)

            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)

    avg_loss = running_loss / len(test_loader)
    accuracy = correct / total

    return avg_loss, accuracy

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = NN4Layer(input_dim, hidden_dim, output_dim).to(device)

print("Model Architecture:\n")
print(model)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params}")

In [ ]:
# Task 5: Start training for 20 epochs:

num_epochs = 20
learning_rate = 0.001

criterion = nn.CrossEntropyLoss()

optimizer = Adam(model.parameters(), learning_rate)


# Run Training
train_losses = []
val_losses = []
val_accuracies = []

print('Starting Training...')
for epoch in range(num_epochs):

    train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)
    val_loss, val_accuracy = validate(model, criterion, test_loader, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}')

print('Training Complete!')





In [ ]:
# Plotting results
plt.figure(figsize=(7, 5))

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here: